In [ ]:
## The intent of the code below is as follows:
# 1. Load a CSV file containing job postings data which have been cleaned.
# 2. Provide a Streamlit web application to visualize and analyze the job postings data.
# 3. The objective is to provide a new investor coming into Singapore
#    (a) median cost per employee for a particular sector.
#    (b) Attractiveness of a sector based on the number of vacancies and applications.



%%writefile app.py
from pathlib import Path
import re

import altair as alt
import pandas as pd
import streamlit as st


st.set_page_config(page_title="2023 Singapore Jobs Outlook", page_icon="💼", layout="wide")

DATE_COLUMNS = [
    "metadata_expiryDate",
    "metadata_newPostingDate",
    "metadata_originalPostingDate",
]

USE_COLUMNS = [
    "metadata_jobPostId",
    "metadata_newPostingDate",
    "metadata_originalPostingDate",
    "metadata_expiryDate",
    "category",
    "employmentTypes",
    "positionLevels",
    "postedCompany_name",
    "title",
    "status_jobStatus",
    "minimumYearsExperience",
    "numberOfVacancies",
    "metadata_totalNumberJobApplication",
    "metadata_totalNumberOfView",
    "salary_minimum",
    "salary_maximum",
    "average_salary",
]


@st.cache_data(show_spinner="Loading job data…")
def load_data(source) -> pd.DataFrame:
    # Read the headers first, then remove accidental leading/trailing spaces.
    # This prevents valid columns from being missed because of whitespace.
    df = pd.read_csv(source, low_memory=False)
    df.columns = df.columns.str.strip()

    missing_columns = [column for column in USE_COLUMNS if column not in df.columns]
    if missing_columns:
        raise ValueError(
            "Missing required CSV columns: " + ", ".join(missing_columns)
        )

    # Retain only the columns used by the dashboard.
    df = df[USE_COLUMNS].copy()

    for column in DATE_COLUMNS:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], errors="coerce")
    return df


def find_default_csv() -> Path | None:
    candidates = [
        Path(__file__).with_name("clean_data_2023.csv"),
        Path.cwd() / "clean_data_2023.csv",
        Path.cwd() / "upload" / "clean_data_2023.csv",
    ]
    return next((path for path in candidates if path.exists()), None)


def select_filter(label: str, values: pd.Series) -> list:
    options = sorted(values.dropna().astype(str).unique().tolist())
    return st.sidebar.multiselect(label, options, placeholder="All")


st.title("💼 2023 Singapore Jobs Outlook")
st.caption("Explore job demand, salaries, applications, and posting trends.")

default_csv = find_default_csv()
uploaded_file = st.sidebar.file_uploader("Use another CSV", type="csv")

if uploaded_file is not None:
    data_source = uploaded_file
elif default_csv is not None:
    data_source = default_csv
else:
    st.info("Place `clean_data_2023(1).csv` beside app.py, or upload it using the sidebar.")
    st.stop()

try:
    jobs = load_data(data_source)
except (ValueError, KeyError) as exc:
    st.error(f"The CSV does not contain the expected job-data columns: {exc}")
    st.stop()

st.sidebar.header("Filters")

valid_dates = jobs["metadata_newPostingDate"].dropna()
if valid_dates.empty:
    date_range = None
else:
    minimum_date = valid_dates.min().date()
    maximum_date = valid_dates.max().date()
    date_range = st.sidebar.date_input(
        "New posting date",
        value=(minimum_date, maximum_date),
        min_value=minimum_date,
        max_value=maximum_date,
    )

category_options = sorted(
    jobs["category"].dropna().str.split(", ").explode().str.strip().unique().tolist()
)
selected_categories = st.sidebar.multiselect("Category", category_options, placeholder="All")
selected_employment = select_filter("Employment type", jobs["employmentTypes"])
selected_levels = select_filter("Position level", jobs["positionLevels"])
selected_statuses = select_filter("Job status", jobs["status_jobStatus"])
company_search = st.sidebar.text_input("Company contains")

filtered = jobs
if date_range and len(date_range) == 2:
    start_date, end_date = pd.Timestamp(date_range[0]), pd.Timestamp(date_range[1])
    filtered = filtered[filtered["metadata_newPostingDate"].between(start_date, end_date)]
if selected_categories:
    category_pattern = "|".join(re.escape(value) for value in selected_categories)
    filtered = filtered[
        filtered["category"].str.contains(category_pattern, case=False, na=False, regex=True)
    ]
if selected_employment:
    filtered = filtered[filtered["employmentTypes"].isin(selected_employment)]
if selected_levels:
    filtered = filtered[filtered["positionLevels"].isin(selected_levels)]
if selected_statuses:
    filtered = filtered[filtered["status_jobStatus"].isin(selected_statuses)]
if company_search:
    filtered = filtered[
        filtered["postedCompany_name"].str.contains(company_search, case=False, na=False)
    ]

st.caption(f"Showing {len(filtered):,} of {len(jobs):,} job postings")

kpi1, kpi2, kpi3, kpi4 = st.columns(4)
kpi1.metric("Job postings", f"{filtered['metadata_jobPostId'].nunique():,}")
kpi2.metric("Vacancies", f"{filtered['numberOfVacancies'].sum():,.0f}")
kpi3.metric("Median salary", f"S${filtered['average_salary'].median():,.0f}")
kpi4.metric("Applications", f"{filtered['metadata_totalNumberJobApplication'].sum():,.0f}")

if filtered.empty:
    st.warning("No records match the current filters.")
    st.stop()

trend_tab, demand_tab, salary_tab, data_tab = st.tabs(
    ["Posting trends", "Job demand", "Salary insights", "Job records"]
)

with trend_tab:
    monthly = (
        filtered.dropna(subset=["metadata_originalPostingDate"])
        .set_index("metadata_originalPostingDate")
        .resample("MS")
        .agg(postings=("metadata_jobPostId", "nunique"), vacancies=("numberOfVacancies", "sum"))
        .reset_index()
    )
    metric_choice = st.radio(
        "Trend measure", ["postings", "vacancies"], horizontal=True, label_visibility="collapsed"
    )
    trend_chart = (
        alt.Chart(monthly)
        .mark_line(point=True)
        .encode(
            x=alt.X("metadata_originalPostingDate:T", title="Original posting month"),
            y=alt.Y(f"{metric_choice}:Q", title=metric_choice.title()),
            tooltip=[
                alt.Tooltip(
                    "metadata_originalPostingDate:T",
                    title="Original posting month",
                    format="%b %Y",
                ),
                metric_choice,
            ],
        )
        .properties(height=420)
        .interactive()
    )
    st.altair_chart(trend_chart, use_container_width=True)

with demand_tab:
    category_vacancies = filtered[["category", "numberOfVacancies"]].copy()
    category_vacancies["category"] = category_vacancies["category"].str.split(", ")
    category_vacancies = category_vacancies.explode("category")
    category_vacancies["category"] = category_vacancies["category"].str.strip()
    category_vacancies = (
        category_vacancies.groupby("category", as_index=False)
        .agg(vacancies=("numberOfVacancies", "sum"))
        .nlargest(15, "vacancies")
    )
    category_chart = (
        alt.Chart(category_vacancies)
        .mark_bar()
        .encode(
            x=alt.X("vacancies:Q", title="Job vacancies"),
            y=alt.Y("category:N", sort="-x", title=None),
            tooltip=["category", alt.Tooltip("vacancies:Q", format=",.0f")],
        )
        .properties(title="Top 15 job categories by vacancies", height=460)
    )
    st.altair_chart(category_chart, use_container_width=True)

    category_demand = filtered[
        ["category", "numberOfVacancies", "metadata_totalNumberJobApplication"]
    ].copy()
    category_demand["category"] = category_demand["category"].str.split(", ")
    category_demand = category_demand.explode("category")
    category_demand["category"] = category_demand["category"].str.strip()
    category_demand = (
        category_demand.groupby("category", as_index=False)
        .agg(
            vacancies=("numberOfVacancies", "sum"),
            applications=("metadata_totalNumberJobApplication", "sum"),
        )
    )
    category_demand = category_demand[category_demand["applications"] > 0].copy()
    category_demand["vacancies_per_application"] = (
        category_demand["vacancies"] / category_demand["applications"]
    )
    category_demand = category_demand.nlargest(15, "vacancies_per_application")

    demand_ratio_chart = (
        alt.Chart(category_demand)
        .mark_bar()
        .encode(
            x=alt.X(
                "vacancies_per_application:Q",
                title="Vacancies per job application",
                axis=alt.Axis(format=".2f"),
            ),
            y=alt.Y("category:N", sort="-x", title=None),
            tooltip=[
                "category",
                alt.Tooltip("vacancies:Q", format=",.0f"),
                alt.Tooltip("applications:Q", format=",.0f"),
                alt.Tooltip(
                    "vacancies_per_application:Q",
                    title="Vacancies ÷ job applications",
                    format=".3f",
                ),
            ],
        )
        .properties(
            title="Top 15 categories by vacancies per job application",
            height=520,
        )
    )
    st.altair_chart(demand_ratio_chart, use_container_width=True)
    st.caption(
        "Calculated as total vacancies divided by total job applications for each category. "
        "Categories with no applications are excluded."
    )

with salary_tab:
    salary_data = filtered[
        filtered["average_salary"].notna() & (filtered["average_salary"] > 0)
    ].copy()
    upper_limit = salary_data["average_salary"].quantile(0.99)
    salary_data = salary_data[salary_data["average_salary"] <= upper_limit]

    left, right = st.columns(2)
    salary_bins = pd.cut(salary_data["average_salary"], bins=40, duplicates="drop")
    salary_histogram = (
        salary_bins.value_counts(sort=False)
        .rename_axis("salary_band")
        .reset_index(name="postings")
    )
    salary_histogram["salary_midpoint"] = salary_histogram["salary_band"].map(
        lambda interval: interval.mid
    ).astype(float)
    histogram = (
        alt.Chart(salary_histogram)
        .mark_bar()
        .encode(
            x=alt.X("salary_midpoint:Q", title="Average monthly salary (S$)"),
            y=alt.Y("postings:Q", title="Job postings"),
            tooltip=[alt.Tooltip("salary_midpoint:Q", title="Salary band midpoint", format=",.0f"), "postings"],
        )
        .properties(title="Salary distribution (up to 99th percentile)", height=430)
    )
    left.altair_chart(histogram, use_container_width=True)

    level_salary = (
        salary_data.groupby("positionLevels", as_index=False)
        .agg(median_salary=("average_salary", "median"), postings=("metadata_jobPostId", "nunique"))
        .sort_values("median_salary", ascending=False)
    )
    level_chart = (
        alt.Chart(level_salary)
        .mark_bar()
        .encode(
            x=alt.X("median_salary:Q", title="Median monthly salary (S$)"),
            y=alt.Y("positionLevels:N", sort="-x", title=None),
            tooltip=[alt.Tooltip("positionLevels:N", title="Position level"), "median_salary", "postings"],
        )
        .properties(title="Median salary by position level", height=430)
    )
    right.altair_chart(level_chart, use_container_width=True)

with data_tab:
    display_columns = [
        "metadata_originalPostingDate", "title", "postedCompany_name", "category",
        "employmentTypes", "positionLevels", "numberOfVacancies",
        "salary_minimum", "salary_maximum", "average_salary", "status_jobStatus",
    ]
    st.dataframe(
        filtered[display_columns]
        .sort_values("metadata_originalPostingDate", ascending=False)
        .head(500),
        use_container_width=True,
        hide_index=True,
        height=520,
    )
    st.caption("The table previews the newest 500 matching records by original posting date.")
    if st.checkbox("Prepare filtered CSV download"):
        st.download_button(
            "Download filtered data",
            data=filtered.to_csv(index=False).encode("utf-8"),
            file_name="filtered_jobs.csv",
            mime="text/csv",
        )



Overwriting app.py
